# Preprocessing
Gérer les valeurs manquantes, supprimer les doublons, normaliser les formats (timestamps, IP). Encoder les variables catégorielles. Livrable : script preprocessing.py.


## Importation des librairies

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

In [2]:
chemin_CICIDS = '../Data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'
chemin_UNSW   = '../Data/UNSW-NB15_1.csv'
chemin_LOGS   = '../Data/cybersecurity_threat_detection_logs.csv'

df_cicids = pd.read_csv(chemin_CICIDS, low_memory=False)
df_unsw   = pd.read_csv(chemin_UNSW, low_memory=False)
df_logs   = pd.read_csv(chemin_LOGS, low_memory=False)


## Dataset CIC-IDS-2017

### 1.1 Aperçu 

In [3]:
print(f"Dimensions initiales : {df_cicids.shape}")
print(f"\nTypes de colonnes :")
print(df_cicids.dtypes.value_counts())

Dimensions initiales : (225745, 79)

Types de colonnes :
int64      54
float64    24
str         1
Name: count, dtype: int64


### 1.2 Nettoyage des noms de colonnes

In [4]:
# Supprimer les espaces dans le noms de colonnes
df_cicids.columns = df_cicids.columns.str.strip()
print(df_cicids.columns.tolist())

['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count

### 1.3 Valeurs manquantes

In [5]:
missing = df_cicids.isnull().sum()
print(f"Nombre de valeurs manquantes : {df_cicids.isnull().sum().sum()}")
df_cicids.dropna(inplace=True) # supprime les lignes avec valeurs manquantes
print(f"Lignes restantes : {len(df_cicids)}")

Valeurs manquantes : 4
Lignes restantes : 225741


In [7]:
# Vérification des valeurs infinies
nb_inf = np.isinf(df_cicids.select_dtypes(include=[np.number])).sum().sum()
print(f"Nombre total de valeurs infinies : {nb_inf}")
df_cicids.replace([np.inf, -np.inf], np.nan, inplace=True)
df_cicids.dropna(inplace=True) # supprime les lignes avec valeurs infinies converties en NaN
print(f"Lignes restantes : {len(df_cicids)}")

Nombre total de valeurs infinies : 0
Lignes restantes : 225711


### 1.4 Suppression des doublons

In [8]:
nb_doublons = df_cicids.duplicated().sum()
print(f"Nombre de doublons détectés : {nb_doublons}")
df_cicids.drop_duplicates(inplace=True) #supprime les doublons
print(f"Lignes restantes : {len(df_cicids)}")


Nombre de doublons détectés : 2629
Lignes restantes : 223082


### 1.5 Correction des valeurs aberrantes

In [9]:
# Supprimer les lignes avec Flow Duration négatif 
nb_negatifs = (df_cicids['Flow Duration'] < 0).sum()
print(f"Nombre de lignes avec Flow Duration négatif : {nb_negatifs}")
df_cicids = df_cicids[df_cicids['Flow Duration'] >= 0] #supprime les lignes avec des valeurs négatives
print(f"Lignes restantes : {len(df_cicids)}")


Lignes avec Flow Duration négatif : 2
Lignes restantes : 223080


### 1.6 Encodage de la variable cible (Label)

In [10]:
# Encoder la colonne Label : BENIGN : 0, attaque : 1
print("Avant encodage :")
print(df_cicids['Label'].value_counts())
df_cicids['Label_encoded'] = df_cicids['Label'].apply(lambda x: 0 if x.strip() == 'BENIGN' else 1)
print("\nAprès encodage :")
print(df_cicids['Label_encoded'].value_counts())

Avant encodage :
Label
DDoS      128014
BENIGN     95066
Name: count, dtype: int64

Après encodage :
Label_encoded
1    128014
0     95066
Name: count, dtype: int64


### 1.7 Résumé du preprocessing CICIDS2017

In [11]:
print(" CICIDS2017")
print(f"Dimensions finales: {df_cicids.shape}")
print(f"Valeurs manquantes: {df_cicids.isnull().sum().sum()}")
print(f"Doublons restants: {df_cicids.duplicated().sum()}")
print(f"Distribution des classes :")
print(df_cicids['Label_encoded'].value_counts())

 CICIDS2017
Dimensions finales: (223080, 80)
Valeurs manquantes: 0
Doublons restants: 0
Distribution des classes :
Label_encoded
1    128014
0     95066
Name: count, dtype: int64



## Dataset UNSW-NB15

### 2.1 Colonnes

In [11]:
colonnes_unsw = [
    'srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur',
    'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service',
    'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb',
    'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len',
    'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt',
    'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl',
    'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src',
    'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm',
    'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'Label'
]
df_unsw = pd.read_csv(chemin_UNSW, names=colonnes_unsw, low_memory=False)

### 2.2 Valeurs manquantes

In [12]:
missing = df_unsw.isnull().sum()
print(f"Nombre de valeurs manquantes : {df_unsw.isnull().sum().sum()}")
print(f"Lignes restantes : {len(df_unsw)}")

Nombre de valeurs manquantes : 677786
Lignes restantes : 700001


In [13]:
df_unsw['attack_cat'] = df_unsw['attack_cat'].fillna('Normal') # remplir attack_cat avant toute suppression
# supprimer les colonnes avec plus de 50% de manquants sauf attack_cat car c'est une information importante
df_unsw.drop(columns=[col for col in df_unsw.columns if df_unsw[col].isnull().mean() > 0.5 and col != 'attack_cat'], inplace=True)
print(f"Nombre de valeurs manquantes attack_cat : {df_unsw['attack_cat'].isnull().sum()}")
print(f"Lignes restantes : {len(df_unsw)}")

Nombre de valeurs manquantes attack_cat : 0
Lignes restantes : 700001


### 2.3 Suppression des doublons

In [19]:
nb_doublons = df_unsw.duplicated().sum()
print(f"Nombre de doublons détectés : {nb_doublons}")
df_unsw.drop_duplicates(inplace=True) #supprime les doublons
print(f"Lignes restantes : {len(df_unsw)}")

Doublons détectés : 0
Lignes restantes : 640788


### 2.4 Normalisation des timestamps

In [14]:
# Stime et Ltime sont des nombres bruts,le nombre de secondes écoulées depuis le 1er janvier 1970 on les convertit en date lisible
df_unsw['Stime_dt'] = pd.to_datetime(df_unsw['Stime'], unit='s', errors='coerce')
df_unsw['Ltime_dt'] = pd.to_datetime(df_unsw['Ltime'], unit='s', errors='coerce')
print(df_unsw[['Stime', 'Stime_dt', 'Ltime', 'Ltime_dt']].head(3)) #vérification


        Stime            Stime_dt       Ltime            Ltime_dt
0  1421927414 2015-01-22 11:50:14  1421927414 2015-01-22 11:50:14
1  1421927414 2015-01-22 11:50:14  1421927414 2015-01-22 11:50:14
2  1421927414 2015-01-22 11:50:14  1421927414 2015-01-22 11:50:14


### 2.5 Normalisation des adresses IP

In [15]:
# Vérification des adresses IP
def ip_valide(ip):
    try:
        parts = str(ip).split('.') # coupe l'IP au niveau des points
        return len(parts) == 4 and all(0 <= int(p) <= 255 for p in parts) # On vérifie qu'il y a bien 4 parties et que chaque nombre est entre 0 et 255
    except:
        return False

nb_invalides = (~df_unsw['srcip'].apply(ip_valide) | ~df_unsw['dstip'].apply(ip_valide)).sum() # on compte le nombre de lignes qui ont au moins une IP invalide
print(f"Nombre d'adresses IP invalides : {nb_invalides}")


Adresses IP invalides : 0


Toutes les  adresses IP sont valides donc il n'y a rien à faire.

### 2.6 Encodage des variables catégorielles

In [16]:
le = LabelEncoder() # crée l'outil qui va faire la conversion texte → chiffre
cols_cat = ['proto', 'state', 'service', 'attack_cat'] # liste des colonnes à encoder
for col in cols_cat:
    if col in df_unsw.columns:  # vérifie que la colonne existe
        df_unsw[col + '_encoded'] = le.fit_transform(df_unsw[col].astype(str)) # crée une nouvelle colonne avec les chiffres
        print(f"{col} : {dict(zip(le.classes_, le.transform(le.classes_)))}")  # affiche quel texte pour quel chiffre

proto : {'3pc': np.int64(0), 'a/n': np.int64(1), 'aes-sp3-d': np.int64(2), 'any': np.int64(3), 'argus': np.int64(4), 'aris': np.int64(5), 'arp': np.int64(6), 'ax.25': np.int64(7), 'bbn-rcc': np.int64(8), 'bna': np.int64(9), 'br-sat-mon': np.int64(10), 'cbt': np.int64(11), 'cftp': np.int64(12), 'chaos': np.int64(13), 'compaq-peer': np.int64(14), 'cphb': np.int64(15), 'cpnx': np.int64(16), 'crtp': np.int64(17), 'crudp': np.int64(18), 'dcn': np.int64(19), 'ddp': np.int64(20), 'ddx': np.int64(21), 'dgp': np.int64(22), 'egp': np.int64(23), 'eigrp': np.int64(24), 'emcon': np.int64(25), 'encap': np.int64(26), 'esp': np.int64(27), 'etherip': np.int64(28), 'fc': np.int64(29), 'fire': np.int64(30), 'ggp': np.int64(31), 'gmtp': np.int64(32), 'gre': np.int64(33), 'hmp': np.int64(34), 'i-nlsp': np.int64(35), 'iatp': np.int64(36), 'ib': np.int64(37), 'icmp': np.int64(38), 'idpr': np.int64(39), 'idpr-cmtp': np.int64(40), 'idrp': np.int64(41), 'ifmp': np.int64(42), 'igmp': np.int64(43), 'igp': np.int6

### 2.7 Résumé du preprocessing UNSW-NB15

In [23]:
print("UNSW-NB15")
print(f"Dimensions finale: {df_unsw.shape}")
print(f"Valeurs manquantes: {df_unsw.isnull().sum().sum()}")
print(f"Doublons restants: {df_unsw.duplicated().sum()}")
print(f"Distribution des classes :")
print(df_unsw['Label'].value_counts())

UNSW-NB15
Dimensions finale: (640788, 55)
Valeurs manquantes: 0
Doublons restants: 0
Distribution des classes :
Label
0    626510
1     14278
Name: count, dtype: int64



## Dataset Logs

### 3.1 Aperçu initial

In [17]:
print(f"Dimensions initiales : {df_logs.shape}")
print(f"\nColonnes : {df_logs.columns.tolist()}")
print(f"\nTypes :")
print(df_logs.dtypes)

Dimensions initiales : (6000000, 10)

Colonnes : ['timestamp', 'source_ip', 'dest_ip', 'protocol', 'action', 'threat_label', 'log_type', 'bytes_transferred', 'user_agent', 'request_path']

Types :
timestamp              str
source_ip              str
dest_ip                str
protocol               str
action                 str
threat_label           str
log_type               str
bytes_transferred    int64
user_agent             str
request_path           str
dtype: object


### 3.2 Valeurs manquantes

In [18]:
print(f"Valeurs manquantes : {df_logs.isnull().sum().sum()}")
print(f"Lignes restantes : {len(df_logs)}")

Valeurs manquantes : 0
Lignes restantes : 6000000


Il n'y a aucune valeur manquante donc il n'y a rien à faire

### 3.3 Suppression des doublons

In [19]:
print(f"Doublons détectés : {df_logs.duplicated().sum()}")
print(f"Lignes restantes : {len(df_logs)}")

Doublons détectés : 0
Lignes restantes : 6000000


Il n'y a aucun doublon don il n'y a rien à faire

### 3.4 Normalisation des timestamps

In [27]:
col_ts = [c for c in df_logs.columns if 'time' in c.lower()] # cherche toutes les colonnes qui contiennent le mot "time"
print(f"Colonnes timestamp détectées : {col_ts}")
for col in col_ts:
    df_logs[col + '_dt']= pd.to_datetime(df_logs[col], errors='coerce') # convertit le texte en datetime pandas
    df_logs[col + '_hour']= df_logs[col + '_dt'].dt.hour  # extrait l'heure 
    df_logs[col + '_weekday'] = df_logs[col + '_dt'].dt.weekday # extrait le jour de la semaine
print(df_logs[[col_ts[0], col_ts[0] + '_dt', col_ts[0] + '_hour']].head(3)) #vérification

Colonnes timestamp détectées : ['timestamp']
             timestamp timestamp_dt  timestamp_hour
0  2024-05-01T00:00:00   2024-05-01               0
1  2024-07-18T00:00:00   2024-07-18               0
2  2024-04-07T00:00:00   2024-04-07               0


### 3.5 Normalisation des adresses IP

In [ ]:
def ip_valide(ip):
    try:
        parts = str(ip).split('.')  # coupe l'IP au niveau des points
        return len(parts) == 4 and all(0 <= int(p) <= 255 for p in parts)  # On vérifie qu'il y a bien 4 parties et que chaque nombre est entre 0 et 255
    except:
        return False

nb_invalides = (~df_logs['source_ip'].apply(ip_valide) | ~df_logs['dest_ip'].apply(ip_valide)).sum()  # on compte le nombre de lignes qui ont au moins une IP invalide
print(f"Nombre d' adresses IP invalides : {nb_invalides}")


Toutes les IP sont valides donc il n'y a rien à faire.

### 3.6 Encodage des variables catégorielles

In [29]:
le = LabelEncoder()# crée l'outil d'encodage
cols_cat = [c for c in df_logs.select_dtypes(include='object').columns if '_dt' not in c] # sélectionne toutes les colonnes texte mais pas date
for col in cols_cat:
    df_logs[col + '_encoded'] = le.fit_transform(df_logs[col].astype(str)) #crée une nouvelle colonne avec les chiffres
    print(f"{col} : {dict(zip(le.classes_, le.transform(le.classes_)))}")# affiche le texte en chiffre

timestamp : {'2024-01-01T00:00:00': np.int64(0), '2024-01-02T00:00:00': np.int64(1), '2024-01-03T00:00:00': np.int64(2), '2024-01-04T00:00:00': np.int64(3), '2024-01-05T00:00:00': np.int64(4), '2024-01-06T00:00:00': np.int64(5), '2024-01-07T00:00:00': np.int64(6), '2024-01-08T00:00:00': np.int64(7), '2024-01-09T00:00:00': np.int64(8), '2024-01-10T00:00:00': np.int64(9), '2024-01-11T00:00:00': np.int64(10), '2024-01-12T00:00:00': np.int64(11), '2024-01-13T00:00:00': np.int64(12), '2024-01-14T00:00:00': np.int64(13), '2024-01-15T00:00:00': np.int64(14), '2024-01-16T00:00:00': np.int64(15), '2024-01-17T00:00:00': np.int64(16), '2024-01-18T00:00:00': np.int64(17), '2024-01-19T00:00:00': np.int64(18), '2024-01-20T00:00:00': np.int64(19), '2024-01-21T00:00:00': np.int64(20), '2024-01-22T00:00:00': np.int64(21), '2024-01-23T00:00:00': np.int64(22), '2024-01-24T00:00:00': np.int64(23), '2024-01-25T00:00:00': np.int64(24), '2024-01-26T00:00:00': np.int64(25), '2024-01-27T00:00:00': np.int64(26)

### 3.7 Résumé du preprocessing Logs

In [ ]:
print("Dataset Log")
print(f"Dimensions finales: {df_logs.shape}")
print(f"Valeurs manquantes: {df_logs.isnull().sum().sum()}")
print(f"Doublons restants: {df_logs.duplicated().sum()}")
print(f"Distribution des classes :")
print(df_logs['threat_label'].value_counts())


## Sauvegarde des datasets nettoyés

In [ ]:
df_cicids.to_csv('../Data/cicids_clean.csv', index=False)
df_unsw.to_csv('../Data/unsw_clean.csv', index=False)
df_logs.to_csv('../Data/logs_clean.csv', index=False)

print("Fichiers sauvegardés :")
print("  → ../Data/cicids_clean.csv")
print("  → ../Data/unsw_clean.csv")
print("  → ../Data/logs_clean.csv")